# 完整实验 Notebook — 48 组实验全流程

> **使用说明**：按顺序运行每个 cell。Phase 1 不需要 GPU，Phase 2/3/4 需要 A100。
>
> **文件结构说明**：
> - GitHub 仓库（`8307-project`）只含代码，**不含数据集**（CSV 太大不放 git）
> - 数据集在 Google Drive 的 `MyDrive/8307/Datasets/`
> - 通过符号链接让代码能找到数据：`/content/Datasets → Drive 中的数据集`
> - 实验结果保存在 `/content/8307-project/results/`，跑完记得备份到 Drive

---
## 0. Environment Setup

In [ ]:
# ====== Step 1: Clone 代码 + 挂载 Drive ======
import os

# clone 代码仓库（只有代码，不含数据集）
if not os.path.exists('/content/8307-project'):
    !git clone https://github.com/zigeng22/8307-project.git /content/8307-project
    print('Repo cloned')
else:
    !cd /content/8307-project && git pull
    print('Repo updated')

# 挂载 Google Drive（数据集在这里）
from google.colab import drive
drive.mount('/content/drive')

# 创建符号链接：让代码能通过 /content/Datasets 找到 Drive 里的数据
link = '/content/Datasets'
src = '/content/drive/MyDrive/8307/Datasets'
assert os.path.exists(src), f'请先上传数据集到 {src}'
if os.path.islink(link):
    os.remove(link)
os.symlink(src, link)
print(f'符号链接: {link} -> {src}')
print(f'数据集文件: {os.listdir(src)}')

In [ ]:
# ====== Step 2: 安装依赖 ======
!pip install -q transformers>=4.43.0 peft>=0.11.0 trl>=0.9.0 datasets accelerate
!pip install -q langchain langchain-community faiss-cpu sentence-transformers
!pip install -q rouge-score bert-score scikit-learn
!pip install -q openai tqdm pandas matplotlib seaborn

In [ ]:
# ====== Step 3: 设置路径 + 检查 GPU ======
import sys, os, torch
sys.path.insert(0, '/content/8307-project')
os.chdir('/content/8307-project')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)')
else:
    print('NO GPU — 跑 API 模型可以继续，跑开源模型需要切换到 A100')

# 验证数据集能正常加载
from data.loader import load_sentiment, load_mentalchat, load_medquad
print(f'Sentiment: {len(load_sentiment())} 条')
print(f'MentalChat: {len(load_mentalchat())} 条')
print(f'MedQuAD: {len(load_medquad())} 条')
print('All data OK ✓')

In [ ]:
# ====== Step 4: 设置 API Key（跑 API 模型时需要，跑开源模型可跳过）======
import os, getpass
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste OpenRouter API Key: ')

# 快速测试
from openai import OpenAI
client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=os.environ['OPENROUTER_API_KEY'])
r = client.chat.completions.create(
    model='deepseek/deepseek-chat-v3-0324',
    messages=[{'role': 'user', 'content': 'Hi'}], max_tokens=5)
print('API OK:', r.choices[0].message.content)

In [ ]:
# ====== Step 5: 登录 HuggingFace（跑开源模型时需要）======
# Llama 是 gated model，需要先在 https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct 申请访问
# 然后在 https://huggingface.co/settings/tokens 创建 token
from huggingface_hub import login
login()  # 会弹出输入框，粘贴你的 HuggingFace token

---
## Phase 1: Baseline（零样本）

每个模型 × 3 个任务。API 模型不需要 GPU，开源模型需要 A100。

**注意**：所有 shell 命令前面必须加 `!`

In [ ]:
# API 模型 baseline（不需要 GPU）
!python experiments/run_baseline.py --model deepseek-v3 --task all
!python experiments/run_baseline.py --model mistral-large --task all

In [ ]:
# 开源模型 baseline（需要 A100）— 每次只跑一个模型，避免显存不足
!python experiments/run_baseline.py --model llama-3.1-8b --task all

In [ ]:
!python experiments/run_baseline.py --model qwen2.5-7b --task all

!python experiments/run_baseline.py --model gemma-2-9b --task all

---
## Phase 2: LoRA 微调（仅开源模型，需要 A100）

用 MentalChat16K 训练集对模型微调，然后评估微调后模型在 3 个任务上的表现。

**每个模型训练约 1-3 小时（A100）**，一次只跑一个模型。

In [ ]:
# Step 1: 微调 Llama（训练约 1-3 小时）
!python finetune/lora_train.py --model llama-3.1-8b

In [ ]:
# Step 2: 评估微调后的 Llama（3 个任务）
!python experiments/run_finetuned.py --model llama-3.1-8b --task all \
    --lora_path ./finetune/checkpoints/llama-3.1-8b

In [ ]:
# 微调 Qwen
!python finetune/lora_train.py --model qwen2.5-7b
!python experiments/run_finetuned.py --model qwen2.5-7b --task all \
    --lora_path ./finetune/checkpoints/qwen2.5-7b

In [ ]:
# 微调 Gemma
!python finetune/lora_train.py --model gemma-2-9b
!python experiments/run_finetuned.py --model gemma-2-9b --task all \
    --lora_path ./finetune/checkpoints/gemma-2-9b

In [ ]:
# 备份 LoRA checkpoint 到 Drive（防止 Colab 断开丢失）
!mkdir -p /content/drive/MyDrive/8307/checkpoints
!cp -r finetune/checkpoints/ /content/drive/MyDrive/8307/checkpoints/
print('Checkpoints backed up to Drive')

---
## Phase 3: RAG 检索增强

先建 FAISS 向量索引（一次性），然后对所有模型跑 Base+RAG 和 Fine-tuned+RAG。

RAG 不需要训练，只是在 prompt 前面加上从 MedQuAD 检索到的相关段落。

In [ ]:
# Step 1: 建 FAISS 向量索引（只需运行一次，约 5 分钟）
!python rag/indexer.py

In [ ]:
# Step 2: Base + RAG — API 模型（不需要 GPU）
!python experiments/run_rag.py --model deepseek-v3 --task all
!python experiments/run_rag.py --model mistral-large --task all

In [ ]:
# Step 3: Base + RAG — 开源模型（需要 A100）
!python experiments/run_rag.py --model llama-3.1-8b --task all
!python experiments/run_rag.py --model qwen2.5-7b --task all
!python experiments/run_rag.py --model gemma-2-9b --task all

In [ ]:
# Step 4: Fine-tuned + RAG（需要先完成 Phase 2 微调）
!python experiments/run_rag.py --model llama-3.1-8b --task all \
    --lora_path ./finetune/checkpoints/llama-3.1-8b
!python experiments/run_rag.py --model qwen2.5-7b --task all \
    --lora_path ./finetune/checkpoints/qwen2.5-7b
!python experiments/run_rag.py --model gemma-2-9b --task all \
    --lora_path ./finetune/checkpoints/gemma-2-9b

---
## 查看结果 + 备份

In [ ]:
# 查看全部实验结果
import json
from pathlib import Path

for config_dir in ['baseline', 'finetuned', 'base_rag', 'finetuned_rag']:
    config_path = Path(f'results/{config_dir}')
    if not config_path.exists():
        continue
    print(f'\n{"="*60}')
    print(f'Config: {config_dir}')
    print(f'{"="*60}')
    for model_dir in sorted(config_path.iterdir()):
        if model_dir.is_dir():
            print(f'\n  {model_dir.name}:')
            for f in sorted(model_dir.glob('*_metrics.json')):
                metrics = json.loads(f.read_text())
                # 只显示关键指标，不显示 predictions
                display = {k: v for k, v in metrics.items() if k != 'predictions'}
                print(f'    {f.stem}: {display}')

In [ ]:
# 备份全部结果到 Drive
!mkdir -p /content/drive/MyDrive/8307/results_backup
!cp -r results/ /content/drive/MyDrive/8307/results_backup/
print('Results backed up to Drive ✓')